# MedMCQA + MedQA: deduplicated train / val / test splits (v3)

One call returns fixed, leak-free splits in one common format:

```python
from medsplits import load_splits, build_messages
splits = load_splits("medmcqa")   # {"train": [...], "val": [...], "test": [...]}
messages = build_messages(splits["train"][0])
```

**How to use this notebook:** run the cells top to bottom. Cell 1 is the only place that decides where data lives.
The code is written to `/content/proj` (a temporary Colab folder), so copy `medsplits/` and `tests/` into the team repo
when you are happy. The data itself goes to `DATA_DIR` (Google Drive), never into git.

**Two-tier deduplication (new in v3)**

| Where | Rule | Why |
|---|---|---|
| Inside train (and across the two train sets) | `exact_key`: same question **and** same options | Removes only true copies. Measured on your run: v1's question-only rule removed 30652 MedMCQA train rows, the exact rule removes 18106. The rest were distinct questions. |
| Train vs val/test, and inside val/test, and between the two datasets' eval sets | `leak_key`: long stems (10+ words) match on the question alone; short stems also need the same options | Strict where it matters: a reworded copy of an eval question must not survive in train. |

Earlier fixes kept from v2: pinned dataset commits (recorded in `stats.json`), a config guard in `load_splits`, and tests that check every row.

**Shared settings (all live in `medsplits/data.py`).** Changing any of them changes the splits, so rebuild and re-share `splits/`.

| Setting | Meaning |
|---|---|
| `DATA_DIR` (env var) | Where data, ids and the HF cache are stored. Default `/shared/data`. |
| `SEED`, `VAL_SIZE` | Shuffle seed and validation size carved from each train set. |
| `SHORT_STEM_WORDS` | `leak_key` threshold: stems shorter than this also need identical options to match. |
| `DEDUPE_SCHEME` | Label of the dedupe rules (two-tier). Bump it if you change how keys are built. |
| `HF_REPOS`, `HF_REVISIONS` | Which datasets, and which commit of each (pinned below). |
| `SPLIT_PRIORITY`, `DATASET_ORDER` | Which copy survives a duplicate (test > val > train; MedQA before MedMCQA). |

## Cell 1: environment

Installs the libraries, mounts Drive (only when running on Colab) and sets `DATA_DIR`.
`DATA_DIR` must be set **before** `medsplits` is imported, because `data.py` reads it once at import time.
Every teammate should point it at the same shared folder.

In [1]:
!pip -q install datasets pytest
import os

DATA_DIR = "/content/drive/MyDrive/shared/data"   # <- the one line to edit per team / machine

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass  # not on Colab: DATA_DIR just has to be a writable folder

os.environ["DATA_DIR"] = DATA_DIR
os.makedirs(DATA_DIR, exist_ok=True)
print("DATA_DIR =", DATA_DIR)

Mounted at /content/drive
DATA_DIR = /content/drive/MyDrive/shared/data


## Cell 2: project folders

Creates the package folder `medsplits/` and the `tests/` folder. The following cells write the Python files with
`%%writefile`; the first one overwrites the file and the rest append (`-a`), so you can re-run the notebook top to
bottom safely.

In [2]:
!mkdir -p /content/proj/medsplits /content/proj/tests

## `data.py`, part 1: imports and shared configuration

Imports only what the module needs, then defines every setting that shapes the splits in one block.
`current_config()` (next cell) snapshots this block, and `build_all` saves the snapshot next to the splits.

- `DATA_DIR`: resolved once at import from the env var, falling back to `/shared/data`.
- `VAL_SIZE`: rows carved out of each dataset's train set for validation.
- `SHORT_STEM_WORDS`: threshold used by `leak_key`.
- `DEDUPE_SCHEME`: a label for the two-tier rules; it is part of the saved config, so splits built under another scheme are rejected.
- `HF_REVISIONS`: pinned to the two commits your v2 build resolved. `None` would mean "latest". After a build, `stats.json`
  records `resolved_revisions`, which must equal these pins.
- `SPLIT_PRIORITY` / `DATASET_ORDER`: decide which copy of a duplicate survives. MedQA goes first so its official test
  set (the source of MedAbstain) is never trimmed by MedMCQA.
- `RESOLVED_REVISIONS`: filled while loading, so the build can record the exact dataset commits used.

In [3]:
%%writefile /content/proj/medsplits/data.py
"""Load MedMCQA and MedQA, convert both to one format, deduplicate, and build fixed splits.

Entry point: load_splits("medmcqa" | "medqa") -> {"train": [...], "val": [...], "test": [...]}
Example format: {"id", "source", "question", "options": {"A": ..., "B": ...}, "answer": "B"}
"""
import json
import os
import random
import re
import string
from collections import Counter
from pathlib import Path

# -------------------------------------------------------------------------------------------------
# Shared configuration. Everything here defines the splits. If you change any of it, run build_all()
# again and re-share the splits folder: load_splits refuses to run on splits built with another config.
# -------------------------------------------------------------------------------------------------
DATA_DIR = Path(os.environ.get("DATA_DIR", "/shared/data"))  # set the env var BEFORE importing
SEED = 42
VAL_SIZE = {"medmcqa": 4000, "medqa": 1000}
SHORT_STEM_WORDS = 10  # leak_key: stems with fewer words must also match on options
DEDUPE_SCHEME = "two-tier-v1"  # exact_key inside train, leak_key against val/test; bump if keys change
HF_REPOS = {"medmcqa": "openlifescienceai/medmcqa", "medqa": "GBaker/MedQA-USMLE-4-options"}
HF_REVISIONS = {  # pinned dataset commits (None would mean "latest")
    "medmcqa": "91c6572c454088bf71b679ad90aa8dffcd0d5868",
    "medqa": "0fb93dd23a7339b6dcd27e241cb9b5eca62d4d18",
}
SPLIT_PRIORITY = ("test", "val", "train")  # when a question repeats, the earliest split keeps it
DATASET_ORDER = ("medqa", "medmcqa")  # tie-break inside a split: MedQA first (protects MedAbstain)
SPLITS = ("train", "val", "test")
LETTERS = string.ascii_uppercase  # option labels by position, so any option count up to 26 works

RESOLVED_REVISIONS = {}  # filled by _load_hf: the dataset commits actually used
_PUNCT = re.compile(r"[^\w\s]")

Writing /content/proj/medsplits/data.py


## `current_config()`

Returns the settings above as a plain dict. `build_all` stores it in `splits/stats.json`; `load_splits` compares the
stored copy with the current one. This is what keeps a team on identical splits: if someone edits `SEED`,
`VAL_SIZE`, the threshold, the repos, the revisions or the priority order and forgets to rebuild, loading fails with a
clear message instead of silently using different data.

In [4]:
%%writefile -a /content/proj/medsplits/data.py


def current_config() -> dict:
    """Snapshot of every setting that changes the splits (JSON-serializable)."""
    return {
        "seed": SEED,
        "val_size": VAL_SIZE,
        "short_stem_words": SHORT_STEM_WORDS,
        "dedupe_scheme": DEDUPE_SCHEME,
        "hf_repos": HF_REPOS,
        "hf_revisions": HF_REVISIONS,
        "split_priority": list(SPLIT_PRIORITY),
        "dataset_order": list(DATASET_ORDER),
    }

Appending to /content/proj/medsplits/data.py


## `normalize(text)`

Turns a question into a comparison form: lowercase, every punctuation character replaced by a space, whitespace runs
collapsed. So `"What is X-ray?"` and `"what is x ray"` compare equal. It is deliberately simple. Changing it changes
which rows count as duplicates, so it is part of the shared configuration in practice.

In [5]:
%%writefile -a /content/proj/medsplits/data.py


def normalize(text: str) -> str:
    """Lowercase, replace punctuation with spaces, collapse whitespace."""
    return " ".join(_PUNCT.sub(" ", text.lower()).split())

Appending to /content/proj/medsplits/data.py


## `exact_key(ex)`, `leak_key(ex)` and `_answer_text(ex)` (two-tier keys, new in v3)

Two different questions are asked about "the same question", so there are two keys.

- **`exact_key`**: normalized question + sorted, normalized options. Two rows match only if they are true copies (a
  reordered copy of the same options still matches). Used to deduplicate **train**. On your run this is the 18106 figure
  from the sweep: removing only these is safe for training, and it keeps distinct questions that share a generic stem.
- **`leak_key`**: long stems (at least `SHORT_STEM_WORDS` words) match on the normalized question alone, so a clinical
  vignette whose distractors were edited still counts as the same question; short stems also include the sorted options,
  so generic stems like "Which of the following is true?" do not collide. Used for everything that protects **evaluation**:
  val/test against train, inside val/test, and between the two datasets' eval sets.

Why two: wrongly dropping a distinct question from train only costs a little training data, while keeping a reworded copy
of an eval question in train corrupts the evaluation. So the strict key guards the eval sets and the exact key cleans train.

`_answer_text` returns the normalized text of the correct option. `dedupe_within` uses it to count removed copies that
disagree on the answer. Under `exact_key` the options are identical, so a disagreement is genuine label noise.

In [6]:
%%writefile -a /content/proj/medsplits/data.py


def _answer_text(ex: dict) -> str:
    """Normalized text of the correct option (to detect duplicates that disagree on the answer)."""
    return normalize(ex["options"][ex["answer"]])


def _options_key(ex: dict) -> str:
    return " | ".join(sorted(normalize(v) for v in ex["options"].values()))


def exact_key(ex: dict) -> str:
    """True-copy identity: same question and same set of options. Used to deduplicate train."""
    return f"{normalize(ex['question'])} || {_options_key(ex)}"


def leak_key(ex: dict) -> str:
    """Strict identity for protecting val/test: long stems match on the question alone."""
    q = normalize(ex["question"])
    if len(q.split()) >= SHORT_STEM_WORDS:
        return q
    return f"{q} || {_options_key(ex)}"

Appending to /content/proj/medsplits/data.py


## `_load_hf(name)` (new in v2)

One place that downloads a dataset. It passes the pinned `revision` (or latest when `None`), caches into
`DATA_DIR/hf_cache` so the download persists on Drive, and records the commit hash that was actually used in
`RESOLVED_REVISIONS`. If the Hub lookup for the hash fails, it records the requested revision instead and carries on.

In [7]:
%%writefile -a /content/proj/medsplits/data.py


def _load_hf(name: str):
    """Download one dataset (pinned revision if set) and remember which commit was used."""
    from datasets import load_dataset

    rev = HF_REVISIONS[name]
    ds = load_dataset(HF_REPOS[name], revision=rev, cache_dir=str(DATA_DIR / "hf_cache"))
    try:
        from huggingface_hub import HfApi

        RESOLVED_REVISIONS[name] = HfApi().dataset_info(HF_REPOS[name], revision=rev).sha
    except Exception:
        RESOLVED_REVISIONS[name] = rev
    return ds

Appending to /content/proj/medsplits/data.py


## `load_medmcqa_raw()`

Converts MedMCQA to the common format and returns `{"train": [...], "test": [...]}`.

- HF `train` becomes our train. HF `validation` becomes our **test**, because HF `test` has no usable answers.
- `cop` (0 to 3) becomes a letter. Rows without a valid `cop` are skipped (none were skipped in the v1 run).
- `id` is `medmcqa-<original uuid>`, so ids never collide with MedQA.
- `options` is a dict built from `opa` to `opd` by position, in the same shape MedAbstain needs (any number of options).

In [8]:
%%writefile -a /content/proj/medsplits/data.py


def load_medmcqa_raw() -> dict:
    """MedMCQA in the common format. Official validation is used as our test (official test has no answers)."""
    ds = _load_hf("medmcqa")
    out = {}
    for hf_split, ours in (("train", "train"), ("validation", "test")):
        rows = []
        for r in ds[hf_split]:
            if r["cop"] not in (0, 1, 2, 3):
                continue
            opts = [r["opa"], r["opb"], r["opc"], r["opd"]]
            rows.append({
                "id": f"medmcqa-{r['id']}",
                "source": "medmcqa",
                "question": r["question"].strip(),
                "options": {LETTERS[i]: str(o).strip() for i, o in enumerate(opts)},
                "answer": LETTERS[r["cop"]],
            })
        out[ours] = rows
    return out

Appending to /content/proj/medsplits/data.py


## `load_medqa_raw()`

Converts MedQA (English, 4 options) to the common format and returns `{"train": [...], "test": [...]}`.

- HF only has train and test, so validation is carved from train later.
- `options` already arrives as a dict keyed A to D; it is sorted and stripped.
- The `assert` fails loudly if an answer letter is not one of the options.
- MedQA has no ids, so `id` is `medqa-<split>-<row index>`. This is stable as long as the pinned revision (and therefore row order) is unchanged, which is one more reason to pin `HF_REVISIONS`.

In [9]:
%%writefile -a /content/proj/medsplits/data.py


def load_medqa_raw() -> dict:
    """MedQA (English, 4 options) in the common format. HF has train and test only."""
    ds = _load_hf("medqa")
    out = {}
    for split in ("train", "test"):
        rows = []
        for i, r in enumerate(ds[split]):
            opts = {k: str(v).strip() for k, v in sorted(r["options"].items())}
            assert r["answer_idx"] in opts, (split, i)
            rows.append({
                "id": f"medqa-{split}-{i}",
                "source": "medqa",
                "question": r["question"].strip(),
                "options": opts,
                "answer": r["answer_idx"],
            })
        out[split] = rows
    return out

Appending to /content/proj/medsplits/data.py


## `dedupe_within(rows, key_fn)`

Removes repeated rows inside one list, keeping the **first** occurrence as decided by `key_fn` (`exact_key` for train,
`leak_key` for test). Returns `(kept_rows, number_removed, number_of_removed_rows_whose_answer_text_differs_from_the_kept_one)`.

The third number counts removed copies that disagree on the correct answer. With `exact_key` the options are identical, so
this is genuine label noise in the source dataset (it was 1983 under the old question-only key, but that number mixed in
different-options rows). It is reported in `stats.json` and does not change which rows are kept.

In [10]:
%%writefile -a /content/proj/medsplits/data.py


def dedupe_within(rows: list, key_fn):
    """Keep the first row per key_fn(row). Returns (kept, n_removed, n_removed_with_conflicting_answer)."""
    first, kept = {}, []
    removed = conflicts = 0
    for r in rows:
        k = key_fn(r)
        if k not in first:
            first[k] = r
            kept.append(r)
            continue
        removed += 1
        if _answer_text(first[k]) != _answer_text(r):
            conflicts += 1
    return kept, removed, conflicts

Appending to /content/proj/medsplits/data.py


## `carve_val(rows, k, seed)`

Takes `k` rows out of a train set to form the validation set. It sorts by `id` first, so the result does not depend on
the order the dataset happens to arrive in, then shuffles with a private `random.Random(seed)` (no global RNG state is
touched). Returns `(train, val)`. Same input and seed always give the same split.

In [11]:
%%writefile -a /content/proj/medsplits/data.py


def carve_val(rows: list, k: int, seed: int = SEED):
    """Deterministically split rows into (train, val) with |val| = k."""
    if not 0 <= k < len(rows):
        raise ValueError(f"val size {k} is not valid for {len(rows)} rows")
    ordered = sorted(rows, key=lambda e: e["id"])
    random.Random(seed).shuffle(ordered)
    return ordered[k:], ordered[:k]

Appending to /content/proj/medsplits/data.py


## `global_dedupe(pools)`

The leakage guard, now two-tier. `pools` is `{dataset: {"train": [...], "val": [...], "test": [...]}}`. Splits are visited
in `SPLIT_PRIORITY` order (test, val, then train) and datasets in `DATASET_ORDER`.

- **Test and val** (both datasets share one `eval_seen` set of `leak_key`s): a row whose `leak_key` was already seen is
  dropped. So a question shared by both test sets stays in MedQA test and is removed from MedMCQA test, and MedQA test is
  never trimmed (MedAbstain's source stays intact).
- **Train**: a row is dropped if its `leak_key` is in `eval_seen` (strict leakage guard, counted as `cross`), or if its
  `exact_key` was already seen in the other dataset's train or earlier in this one (a true copy, counted as `cross_dup`).
  Two train rows that merely share a long stem but have different options are both kept.

An `assert` checks that `train` is last in `SPLIT_PRIORITY`, because the train rule needs the complete eval key set.
Returns `(final_pools, removal_counts)`; every count key is present (0 if none).

Side effect to know about: validation is carved *before* this pass, so a val row that matches a test question is dropped
too. The final val set can therefore be a few rows smaller than `VAL_SIZE`; the exact sizes are in `stats.json`.

In [12]:
%%writefile -a /content/proj/medsplits/data.py


def global_dedupe(pools: dict):
    """Two-tier cross-split pass: leak_key protects val/test, exact_key cleans train."""
    assert SPLIT_PRIORITY[-1] == "train", "train must be processed after every eval split"
    eval_seen, train_seen = set(), set()
    final = {name: {} for name in pools}
    removed = Counter({f"{n}/{s}/cross": 0 for n in pools for s in SPLITS})
    removed.update({f"{n}/train/cross_dup": 0 for n in pools})
    for split in SPLIT_PRIORITY:
        for name in DATASET_ORDER:
            kept = []
            for ex in pools[name][split]:
                lk = leak_key(ex)
                if lk in eval_seen:
                    removed[f"{name}/{split}/cross"] += 1
                    continue
                if split == "train":
                    ek = exact_key(ex)
                    if ek in train_seen:
                        removed[f"{name}/train/cross_dup"] += 1
                        continue
                    train_seen.add(ek)
                else:
                    eval_seen.add(lk)
                kept.append(ex)
            final[name][split] = kept
    return final, removed

Appending to /content/proj/medsplits/data.py


## `build_all(out_dir, verbose)`

Runs the whole pipeline and writes everything to disk. Order matters:

1. load both datasets (common format);
2. `dedupe_within` on each list: train with `exact_key`, test with `leak_key`;
3. `carve_val` from the deduplicated train;
4. `global_dedupe` across splits and datasets;
5. write `splits/<dataset>.json` (the id lists, the source of truth), `processed/<dataset>.jsonl` (the records) and
   `splits/stats.json` (config, resolved dataset commits, raw and final sizes, removal counts, answer-conflict counts).

Deduplicating train *before* carving val prevents one copy of a duplicate landing in val and another in train.
`out_dir` defaults to `DATA_DIR`; the determinism test passes temporary folders.

In [13]:
%%writefile -a /content/proj/medsplits/data.py


def build_all(out_dir=None, verbose: bool = True) -> dict:
    """Build every split from scratch and write ids, records and stats to out_dir."""
    out_dir = Path(out_dir or DATA_DIR)
    (out_dir / "splits").mkdir(parents=True, exist_ok=True)
    (out_dir / "processed").mkdir(parents=True, exist_ok=True)

    raw = {"medmcqa": load_medmcqa_raw(), "medqa": load_medqa_raw()}
    raw_sizes = {n: {s: len(v) for s, v in parts.items()} for n, parts in raw.items()}

    removed, conflicts, pools = Counter(), {}, {}
    for name, parts in raw.items():
        train, n_removed, n_conf = dedupe_within(parts["train"], exact_key)
        removed[f"{name}/train/within"], conflicts[f"{name}/train"] = n_removed, n_conf
        test, n_removed, n_conf = dedupe_within(parts["test"], leak_key)
        removed[f"{name}/test/within"], conflicts[f"{name}/test"] = n_removed, n_conf
        train, val = carve_val(train, VAL_SIZE[name], SEED)
        pools[name] = {"train": train, "val": val, "test": test}

    final, cross_removed = global_dedupe(pools)
    removed.update(cross_removed)

    stats = {
        "config": json.loads(json.dumps(current_config())),
        "resolved_revisions": dict(RESOLVED_REVISIONS),
        "raw_sizes": raw_sizes,
        "removed": dict(removed),
        "duplicate_answer_conflicts": conflicts,
        "sizes": {n: {s: len(final[n][s]) for s in SPLITS} for n in final},
    }

    for name, parts in final.items():
        ids = {s: [e["id"] for e in parts[s]] for s in SPLITS}
        (out_dir / "splits" / f"{name}.json").write_text(json.dumps(ids))
        with open(out_dir / "processed" / f"{name}.jsonl", "w", encoding="utf-8") as f:
            for s in SPLITS:
                for e in parts[s]:
                    f.write(json.dumps(e, ensure_ascii=False) + "\n")
    (out_dir / "splits" / "stats.json").write_text(json.dumps(stats, indent=2))
    if verbose:
        print(json.dumps(stats, indent=2))
    return stats

Appending to /content/proj/medsplits/data.py


## `load_splits(name, data_dir, check_config)`: the entry point

`load_splits("medmcqa")` or `load_splits("medqa")` returns `{"train": [...], "val": [...], "test": [...]}`, each a list
of common-format dicts.

- If the saved splits do not exist yet, it builds them first.
- It reads the id lists (the source of truth) and looks the records up in `processed/<name>.jsonl`.
- **Config guard:** the configuration stored with the splits must equal the current one; otherwise it raises
  `RuntimeError` telling you to rebuild or revert. Pass `check_config=False` only for debugging.
- It never re-derives splits from random state, so everyone reading the same folder gets identical data.

In [14]:
%%writefile -a /content/proj/medsplits/data.py


def load_splits(name: str, data_dir=None, check_config: bool = True) -> dict:
    """Return {"train": [...], "val": [...], "test": [...]} for "medmcqa" or "medqa"."""
    if name not in HF_REPOS:
        raise ValueError(f"unknown dataset {name!r}; expected one of {sorted(HF_REPOS)}")
    data_dir = Path(data_dir or DATA_DIR)
    split_file = data_dir / "splits" / f"{name}.json"
    stats_file = data_dir / "splits" / "stats.json"
    if not split_file.exists() or not stats_file.exists():
        build_all(data_dir)

    if check_config:
        saved = json.loads(stats_file.read_text())["config"]
        now = json.loads(json.dumps(current_config()))
        if saved != now:
            raise RuntimeError(
                "Saved splits were built with a different configuration than the current code.\n"
                f"  saved: {saved}\n  now:   {now}\n"
                "Revert the setting, or run build_all() and share the new splits folder."
            )

    ids = json.loads(split_file.read_text())
    with open(data_dir / "processed" / f"{name}.jsonl", encoding="utf-8") as f:
        by_id = {e["id"]: e for e in map(json.loads, f)}
    return {s: [by_id[i] for i in ids[s]] for s in SPLITS}

Appending to /content/proj/medsplits/data.py


## `prompts.py`

The single definition of the prompt, imported by the model wrapper (#5) and training (#7).
Edit the wording here only, and bump `PROMPT_VERSION` so results can be traced to a prompt.

- `INSTRUCTION`: the final line; `{letters}` is filled from the example's own options, so 4 or 6 options both work.
- `format_question(ex)`: question, blank line, lettered options in letter order, blank line, instruction. Option order in the
  dict does not matter because options are sorted by letter.
- `build_messages(ex, include_answer=False)`: returns chat messages `[{"role": "user", ...}]`. With
  `include_answer=True` it appends an assistant message containing only the correct letter (for supervised training).

In [15]:
%%writefile /content/proj/medsplits/prompts.py
"""The one and only prompt format. Used by the model wrapper (#5) and training (#7)."""

PROMPT_VERSION = "v1"  # bump on any wording change
INSTRUCTION = "Answer with only the letter ({letters})."


def format_question(ex: dict) -> str:
    """Question, lettered options, and the answer-format instruction as one string."""
    options = dict(sorted(ex["options"].items()))
    lines = [ex["question"].strip(), ""]
    lines += [f"{letter}. {text}" for letter, text in options.items()]
    lines += ["", INSTRUCTION.format(letters=", ".join(options))]
    return "\n".join(lines)


def build_messages(ex: dict, include_answer: bool = False) -> list:
    """Chat messages for one example; add the gold letter as an assistant turn for training."""
    messages = [{"role": "user", "content": format_question(ex)}]
    if include_answer:
        messages.append({"role": "assistant", "content": ex["answer"]})
    return messages

Writing /content/proj/medsplits/prompts.py


## `__init__.py`

Re-exports the public names so callers can write `from medsplits import load_splits, build_messages`.

In [16]:
%%writefile /content/proj/medsplits/__init__.py
from .data import build_all, exact_key, leak_key, load_splits, normalize
from .prompts import PROMPT_VERSION, build_messages, format_question

Writing /content/proj/medsplits/__init__.py


## Tests

**`tests/test_data.py`**

| Test | What it guarantees |
|---|---|
| `test_no_eval_question_in_any_train` | No val/test question of either dataset appears in any train set, judged by the strict `leak_key` (the required leakage check). |
| `test_eval_sets_have_no_duplicates` | `leak_key`s are unique across all val and test rows of both datasets. |
| `test_train_has_no_exact_duplicates` | `exact_key`s are unique across both train sets. |
| `test_ids_unique` | Ids are unique across everything. |
| `test_format_all_rows` | Every row: contiguous letters from `A`, answer is an option, correct `source` and id prefix, non-empty question. |
| `test_medqa_test_is_untouched` | MedQA test equals the official test row for row, so MedAbstain's source is intact. |
| `test_deterministic` | Building twice gives byte-identical id files. |
| `test_config_guard` | Changing a setting without rebuilding makes `load_splits` raise. |

**`tests/test_dedupe_logic.py`** (offline, instant, no data needed) checks the key functions and the two-tier pass on small
hand-made examples: same long stem with different options stays in train, a reworded copy of an eval question is removed from
train, true copies across the two train sets are removed, MedQA test wins over MedMCQA test, and generic short stems survive.

**`tests/test_prompts.py`** checks 4 and 6 option prompts, option-order independence and the message structure.

The data tests use the real splits in `DATA_DIR` (built on first use) and take about a minute because
`test_deterministic` rebuilds twice.

In [17]:
%%writefile /content/proj/tests/test_data.py
import pytest

from medsplits import data
from medsplits.data import LETTERS, SPLITS, build_all, exact_key, leak_key, load_medqa_raw, load_splits

NAMES = ["medmcqa", "medqa"]


@pytest.fixture(scope="module")
def splits():
    return {n: load_splits(n) for n in NAMES}


def test_no_eval_question_in_any_train(splits):
    train_keys = {leak_key(e) for n in NAMES for e in splits[n]["train"]}
    for n in NAMES:
        for sp in ("val", "test"):
            leaked = [e["id"] for e in splits[n][sp] if leak_key(e) in train_keys]
            assert not leaked, (n, sp, leaked[:5])


def test_eval_sets_have_no_duplicates(splits):
    keys = [leak_key(e) for n in NAMES for sp in ("val", "test") for e in splits[n][sp]]
    assert len(keys) == len(set(keys))


def test_train_has_no_exact_duplicates(splits):
    keys = [exact_key(e) for n in NAMES for e in splits[n]["train"]]
    assert len(keys) == len(set(keys))


def test_ids_unique(splits):
    ids = [e["id"] for n in NAMES for sp in SPLITS for e in splits[n][sp]]
    assert len(ids) == len(set(ids))


def test_format_all_rows(splits):
    for n in NAMES:
        for sp in SPLITS:
            for e in splits[n][sp]:
                letters = list(e["options"])
                assert letters == list(LETTERS[: len(letters)]), e["id"]
                assert len(letters) >= 2, e["id"]
                assert e["answer"] in e["options"], e["id"]
                assert e["source"] == n and e["id"].startswith(n + "-"), e["id"]
                assert e["question"].strip(), e["id"]


def test_medqa_test_is_untouched(splits):
    n_raw = len(load_medqa_raw()["test"])
    assert {e["id"] for e in splits["medqa"]["test"]} == {f"medqa-test-{i}" for i in range(n_raw)}


def test_deterministic(tmp_path):
    build_all(tmp_path / "a", verbose=False)
    build_all(tmp_path / "b", verbose=False)
    for n in NAMES:
        a = (tmp_path / "a" / "splits" / f"{n}.json").read_text()
        b = (tmp_path / "b" / "splits" / f"{n}.json").read_text()
        assert a == b


def test_config_guard(monkeypatch):
    load_splits("medqa")  # fine with the current config
    monkeypatch.setattr(data, "SEED", data.SEED + 1)
    with pytest.raises(RuntimeError):
        load_splits("medqa")

Writing /content/proj/tests/test_data.py


In [18]:
%%writefile /content/proj/tests/test_dedupe_logic.py
from medsplits import data
from medsplits.data import dedupe_within, exact_key, global_dedupe, leak_key

LONG = "A 45-year-old man presents with chest pain radiating to the left arm and diaphoresis lasting an hour"


def ex(id_, q, opts, ans="A", source="medmcqa"):
    return {"id": id_, "source": source, "question": q,
            "options": dict(zip("ABCDEF", opts)), "answer": ans}


def pools(**kw):
    base = {n: {"train": [], "val": [], "test": []} for n in ("medmcqa", "medqa")}
    for key, rows in kw.items():
        name, split = key.split("__")
        base[name][split] = rows
    return base


def test_keys():
    a = ex("1", LONG, ["w", "x", "y", "z"])
    b = ex("2", LONG.upper() + "?", ["z", "y", "x", "w"])          # punctuation, case, option order
    c = ex("3", LONG, ["p", "q", "r", "s"])                         # same stem, different options
    assert exact_key(a) == exact_key(b) and exact_key(a) != exact_key(c)
    assert leak_key(a) == leak_key(b) == leak_key(c)                # long stem: options ignored
    s1, s2 = ex("4", "Which is true?", ["a", "b", "c", "d"]), ex("5", "Which is true?", ["e", "f", "g", "h"])
    assert leak_key(s1) != leak_key(s2)                             # short stem: options matter


def test_train_keeps_same_stem_with_different_options():
    a, c = ex("1", LONG, ["w", "x", "y", "z"]), ex("3", LONG, ["p", "q", "r", "s"])
    final, removed = global_dedupe(pools(medmcqa__train=[a, c]))
    assert [e["id"] for e in final["medmcqa"]["train"]] == ["1", "3"]
    assert sum(removed.values()) == 0


def test_reworded_eval_question_is_removed_from_train():
    train = [ex("1", LONG, ["p", "q", "r", "s"]), ex("2", "Another stem entirely about renal physiology and fluid balance in adults", ["a", "b", "c", "d"])]
    test = [ex("9", LONG, ["w", "x", "y", "z"])]                    # same stem, different options
    final, removed = global_dedupe(pools(medmcqa__train=train, medmcqa__test=test))
    assert [e["id"] for e in final["medmcqa"]["train"]] == ["2"]
    assert removed["medmcqa/train/cross"] == 1


def test_true_copies_across_train_sets_are_removed():
    a = ex("1", LONG, ["w", "x", "y", "z"], source="medqa")
    b = ex("2", LONG, ["z", "y", "x", "w"])
    final, removed = global_dedupe(pools(medqa__train=[a], medmcqa__train=[b]))
    assert len(final["medqa"]["train"]) == 1 and len(final["medmcqa"]["train"]) == 0
    assert removed["medmcqa/train/cross_dup"] == 1


def test_medqa_test_wins_over_medmcqa_test():
    q = ex("q", LONG, ["a", "b", "c", "d"], source="medqa")
    m = ex("m", LONG, ["a", "b", "c", "d"])
    final, removed = global_dedupe(pools(medqa__test=[q], medmcqa__test=[m]))
    assert len(final["medqa"]["test"]) == 1 and len(final["medmcqa"]["test"]) == 0
    assert removed["medmcqa/test/cross"] == 1


def test_dedupe_within_counts_conflicts():
    a = ex("1", LONG, ["w", "x", "y", "z"], "A")
    b = ex("2", LONG, ["w", "x", "y", "z"], "B")                    # exact copy, different answer
    kept, removed, conflicts = dedupe_within([a, b], exact_key)
    assert [e["id"] for e in kept] == ["1"] and removed == 1 and conflicts == 1


def test_train_must_be_last_in_priority(monkeypatch):
    monkeypatch.setattr(data, "SPLIT_PRIORITY", ("train", "val", "test"))
    try:
        global_dedupe(pools())
    except AssertionError:
        return
    raise AssertionError("expected the priority assertion to fire")

Writing /content/proj/tests/test_dedupe_logic.py


In [19]:
%%writefile /content/proj/tests/test_prompts.py
from medsplits import build_messages, format_question

SIX = {
    "id": "x-1", "source": "x", "question": "Which drug?",
    "options": {"A": "a", "B": "b", "C": "c", "D": "d", "E": "e", "F": "f"}, "answer": "E",
}
FOUR = {**SIX, "options": {k: v for k, v in SIX["options"].items() if k in "ABCD"}, "answer": "B"}


def test_four_and_six_options():
    assert format_question(FOUR).endswith("Answer with only the letter (A, B, C, D).")
    six = format_question(SIX)
    assert "F. f" in six and six.endswith("(A, B, C, D, E, F).")


def test_option_order_does_not_matter():
    shuffled = {**SIX, "options": dict(reversed(list(SIX["options"].items())))}
    assert format_question(shuffled) == format_question(SIX)


def test_messages():
    assert build_messages(FOUR) == [{"role": "user", "content": format_question(FOUR)}]
    assert build_messages(FOUR, include_answer=True)[-1] == {"role": "assistant", "content": "B"}

Writing /content/proj/tests/test_prompts.py


## Cell: build the splits

`build_all()` downloads both datasets (about 90 MB, cached under `DATA_DIR/hf_cache`), builds the splits and prints the
statistics. What to expect from your earlier run (same pinned commits):

- MedMCQA `train/within` should be about 18106 (true copies only; v2 removed 21542, v1 removed 30652), so final MedMCQA train
  should be about 160.7k rows (182822 - 18106 - 4000, minus a handful of `cross` removals).
- MedQA numbers unchanged: 9176 / 1000 / 1273, with MedQA test intact.
- `resolved_revisions` must equal the pins in `HF_REVISIONS`. If it does not, the pin was not applied.

In [20]:
import os, sys
sys.path.insert(0, "/content/proj")

from medsplits import build_all, load_splits, build_messages
stats = build_all()

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

{
  "config": {
    "seed": 42,
    "val_size": {
      "medmcqa": 4000,
      "medqa": 1000
    },
    "short_stem_words": 10,
    "dedupe_scheme": "two-tier-v1",
    "hf_repos": {
      "medmcqa": "openlifescienceai/medmcqa",
      "medqa": "GBaker/MedQA-USMLE-4-options"
    },
    "hf_revisions": {
      "medmcqa": "91c6572c454088bf71b679ad90aa8dffcd0d5868",
      "medqa": "0fb93dd23a7339b6dcd27e241cb9b5eca62d4d18"
    },
    "split_priority": [
      "test",
      "val",
      "train"
    ],
    "dataset_order": [
      "medqa",
      "medmcqa"
    ]
  },
  "resolved_revisions": {
    "medmcqa": "91c6572c454088bf71b679ad90aa8dffcd0d5868",
    "medqa": "0fb93dd23a7339b6dcd27e241cb9b5eca62d4d18"
  },
  "raw_sizes": {
    "medmcqa": {
      "train": 182822,
      "test": 4183
    },
    "medqa": {
      "train": 10178,
      "test": 1273
    }
  },
  "removed": {
    "medmcqa/train/within": 18106,
    "medmcqa/test/within": 15,
    "medqa/train/within": 0,
    "medqa/test/within": 0,


## Cell: look at the result

Prints the split sizes for both datasets and one example in the common format together with the exact prompt the model will see.

In [21]:
for name in ("medmcqa", "medqa"):
    s = load_splits(name)
    print(name, {k: len(v) for k, v in s.items()})

example = load_splits("medqa")["val"][0]
print(example)
print()
for m in build_messages(example):
    print(f"[{m['role']}]\n{m['content']}")

medmcqa {'train': 160536, 'val': 3998, 'test': 4168}
medqa {'train': 9178, 'val': 1000, 'test': 1273}
{'id': 'medqa-train-4439', 'source': 'medqa', 'question': 'A 33-year-old man presents to the emergency department with a fever and fatigue. He states that he has not felt well since he returned from a hiking trip in Alabama. He is generally healthy and has no other medical conditions. His temperature is 101°F (38.3°C), blood pressure is 127/85 mmHg, pulse is 108/min, respirations are 14/min, and oxygen saturation is 99% on room air. Physical exam including a full dermatologic inspection is unremarkable. Laboratory studies are ordered as seen below.\n\nHemoglobin: 13 g/dL\nHematocrit: 39%\nLeukocyte count: 2,200/mm^3 with normal differential\nPlatelet count: 77,000/mm^3\n\nSerum:\nNa+: 139 mEq/L\nCl-: 100 mEq/L\nK+: 4.3 mEq/L\nHCO3-: 24 mEq/L\nBUN: 19 mg/dL\nGlucose: 98 mg/dL\nCreatinine: 1.3 mg/dL\nCa2+: 10.2 mg/dL\nAST: 92 U/L\nALT: 100 U/L\n\nWhich of the following is the most likely

## Cell: run the tests

Runs both test files. All tests must pass before the splits are shared. Takes about a minute.

In [22]:
!cd /content/proj && DATA_DIR={DATA_DIR} python -m pytest -q tests

..................                                                       [100%]
18 passed in 89.72s (0:01:29)


## Cell: reference sweep, how many MedMCQA train rows each key removes

This cell only explains the design; the pipeline does not use it. On your run it printed 30652 (question only, v1), 27165,
23672, 21542 (threshold 10, v2), 20168, 19226 and 18106 (question + options, now `exact_key` for train). Rerun it any time
to see the same numbers.

It compares how many MedMCQA train rows would be removed under different keys:

- threshold `0`: question text only (the v1 rule, expected to reproduce 30652);
- thresholds `6` to `15`: question-only for stems at least that many words long, question + options for shorter stems;
- threshold `1000000`: question + options for everything (exact duplicates only).

The second output shows where the question-only removals come from, by stem length. Reading it:

- The drop from 30652 to 18106 is distinct questions that share a stem but have different options; v1 wrongly discarded them.
- The remaining 18106 are true copies, about 10% of MedMCQA train.
- The gap between 21542 (threshold 10) and 18106 is long stems with different options. In train these are now kept (two-tier);
  against val/test they are still treated as leaks (the strict `leak_key`).
- The most repeated short stems are printed so you can eyeball them.

In [23]:
from collections import Counter, defaultdict
from medsplits.data import load_medmcqa_raw, normalize

rows = load_medmcqa_raw()["train"]

def n_removed(threshold):
    seen, n = set(), 0
    for e in rows:
        q = normalize(e["question"])
        key = q if len(q.split()) >= threshold else q + " || " + " | ".join(sorted(normalize(v) for v in e["options"].values()))
        if key in seen:
            n += 1
        else:
            seen.add(key)
    return n

print("threshold -> rows removed (0 = question-only, 1000000 = question+options)")
for t in (0, 6, 8, 10, 12, 15, 1_000_000):
    print(f"{t:>8} -> {n_removed(t)}")

groups = defaultdict(list)
for e in rows:
    groups[normalize(e["question"])].append(e)
by_len = Counter()
for q, g in groups.items():
    w = len(q.split())
    bucket = "<5 words" if w < 5 else "5-9 words" if w < 10 else "10-19 words" if w < 20 else "20+ words"
    by_len[bucket] += len(g) - 1
print("\nrows removed by question-only rule, by stem length:", dict(by_len))

print("\nmost repeated stems with fewer than 10 words:")
short = sorted(((len(g), q) for q, g in groups.items() if len(q.split()) < 10), reverse=True)[:8]
for count, q in short:
    n_opt_sets = len({tuple(sorted(normalize(v) for v in e["options"].values())) for e in groups[q]})
    print(f"  x{count} ({n_opt_sets} distinct option sets): {q}")

threshold -> rows removed (0 = question-only, 1000000 = question+options)
       0 -> 30652
       6 -> 27165
       8 -> 23672
      10 -> 21542
      12 -> 20168
      15 -> 19226
 1000000 -> 18106

rows removed by question-only rule, by stem length: {'10-19 words': 8000, '5-9 words': 17312, '20+ words': 1826, '<5 words': 3514}

most repeated stems with fewer than 10 words:
  x13 (11 distinct option sets): call exner bodies are seen in
  x8 (5 distinct option sets): schiller duval bodies are seen in
  x8 (6 distinct option sets): hair on end appearance is seen in
  x8 (8 distinct option sets): all of the following statements are true except
  x8 (7 distinct option sets): all are true except
  x7 (2 distinct option sets): whip lash injury is caused due to
  x7 (6 distinct option sets): which of the following is not a prodrug
  x7 (4 distinct option sets): the term id was coined by


## Cell: numbers for the README

Prints a Markdown table of split sizes, the removed-duplicate counts, the number of duplicate pairs with conflicting
answers, and the dataset commits used. Paste the output into the project README (the "Done when" item).

In [24]:
import json
from pathlib import Path

s = json.loads((Path(os.environ["DATA_DIR"]) / "splits" / "stats.json").read_text())
print("| dataset | raw train | raw test | final train | val | test |")
print("|---|---|---|---|---|---|")
for n, d in s["sizes"].items():
    r = s["raw_sizes"][n]
    print(f"| {n} | {r['train']} | {r['test']} | {d['train']} | {d['val']} | {d['test']} |")
print("\nRemoved duplicates (within = copies inside one list; cross = matched an eval question; cross_dup = true copy across the two train sets):")
for k, v in s["removed"].items():
    print(f"- {k}: {v}")
print("\nRemoved duplicates whose answer text disagreed with the kept copy:", s["duplicate_answer_conflicts"])
print("Dataset commits used:", s["resolved_revisions"])
print("Seed:", s["config"]["seed"], "| val sizes:", s["config"]["val_size"], "| short-stem threshold:", s["config"]["short_stem_words"])

| dataset | raw train | raw test | final train | val | test |
|---|---|---|---|---|---|
| medmcqa | 182822 | 4183 | 160536 | 3998 | 4168 |
| medqa | 10178 | 1273 | 9178 | 1000 | 1273 |

Removed duplicates (within = copies inside one list; cross = matched an eval question; cross_dup = true copy across the two train sets):
- medmcqa/train/within: 18106
- medmcqa/test/within: 15
- medqa/train/within: 0
- medqa/test/within: 0
- medmcqa/train/cross: 180
- medmcqa/val/cross: 2
- medmcqa/test/cross: 0
- medqa/train/cross: 0
- medqa/val/cross: 0
- medqa/test/cross: 0
- medmcqa/train/cross_dup: 0
- medqa/train/cross_dup: 0

Removed duplicates whose answer text disagreed with the kept copy: {'medmcqa/train': 400, 'medmcqa/test': 2, 'medqa/train': 0, 'medqa/test': 0}
Dataset commits used: {'medmcqa': '91c6572c454088bf71b679ad90aa8dffcd0d5868', 'medqa': '0fb93dd23a7339b6dcd27e241cb9b5eca62d4d18'}
Seed: 42 | val sizes: {'medmcqa': 4000, 'medqa': 1000} | short-stem threshold: 10


## Before sharing

1. Copy `medsplits/` and `tests/` from `/content/proj` into the team repo (Colab's `/content` is wiped when the session ends).
2. The dataset commits are already pinned in `HF_REVISIONS`; confirm `resolved_revisions` in `stats.json` equals them.
3. Share the `DATA_DIR` folder (it holds `splits/`, `processed/` and `hf_cache/`) and add it to `.gitignore`; commit only the code.
4. Everyone sets `DATA_DIR` to that folder before importing `medsplits`.

In [25]:
import hashlib, os, pathlib
for p in sorted((pathlib.Path(os.environ["DATA_DIR"]) / "splits").glob("*.json")):
    if p.name != "stats.json":
        print(p.name, hashlib.sha256(p.read_bytes()).hexdigest())

medmcqa.json 276f02ced18882ed75e9826abec5ce444551112385669fccee133f4400f5837e
medqa.json 6e755d5bbd8c5c256a86e701a0557e1cb8d81e984f6abf7f6d6f57c550ffbc2c
